In [1]:
# -*- coding: utf-8 -*-
"""
Per-trial Canonical Correlation Analysis (CCA)
=============================================

For every *single* trial, we compute the **maximal canonical correlation**
between the multichannel EEG (theta-band power) and pupil-diameter time
series, allowing for small temporal lags (±1 s in 100 ms steps).

Changes compared to `Temp_Shifts_per_trial_clara.py`
----------------------------------------------------
* **CCA instead of per-electrode Pearson correlations**.
* Preserves the original normalisation strategy:
  - EEG: centre each channel and scale so that the *total* variance across
    **time x channels** equals 1 ↝ keeps relative channel amplitudes.
  - Pupil: trial-wise z-score (mean 0, sd 1).
* Saves one row per *(subject, condition, load, epoch)* with the best
  correlation and lag.

Output
------
`trial_level_cca.csv`  -  columns:
    subject, condition (memory/control), load (05/09/13), epoch (file name),
    r_max (canonical corr.), lag_ms (best EEG→pupil lag, ms)

Requires
--------
* scikit-learn ≥ 1.0  (for `sklearn.cross_decomposition.CCA`)
* pandas, numpy
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler


In [ ]:
###############################################################################
# User settings
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")
OUT_FILE   = Path("trial_level_cca.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 98), [32, 37, 53, 61, 66, 78, 84, 94, 96])
FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

SHIFTS = np.arange(-10, 11)   # ±1 s @10 Hz  →  in *samples*
WIN_OFFSET = 20               # discard first/last 200 ms (20 samples) for padding safety
###############################################################################


def normalise_eeg(matrix: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so \sum_{t,c} x² = 1."""
    centred = matrix - matrix.mean(axis=0, keepdims=True)
    var_total = np.mean(centred ** 2)
    return centred / np.sqrt(var_total)


def best_cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> tuple[float, int]:
    """Return (r_max, best_lag_ms) for one trial."""
    best_r, best_shift = -np.inf, 0

    # window of interest (to avoid circular‑shift artefacts)
    T = min(len(pupil), len(eeg))
    win = slice(WIN_OFFSET, T - WIN_OFFSET)

    # z‑score pupil once outside the loop
    pupil_z = (pupil - pupil.mean()) / pupil.std(ddof=0)
    pupil_z = pupil_z[win].reshape(-1, 1)

    for s in SHIFTS:
        eeg_shifted = np.roll(eeg, s, axis=0)[win]

        # CCA with 1 component (degenerates to optimal regression)
        cca = CCA(n_components=1, max_iter=1000)
        cca.fit(eeg_shifted, pupil_z)

        c_weights_eeg, c_weights_pupil = cca.x_weights_.flatten(), cca.y_weights_.flatten()
        combined_eeg = eeg_shifted @ c_weights_eeg
        combined_pupil = pupil_z * c_weights_pupil # Since pupil has 1D, just multiply

        # Calculate peraron correlation between the combinaed EEG and pupil data
        r = np.corrcoef(combined_eeg, combined_pupil.flatten())[0, 1]
        #u, v = cca.transform(eeg_shifted, pupil_z)
        #r = np.corrcoef(u[:, 0], v[:, 0])[0, 1]

        if r > best_r:
            best_r, best_shift = r, s * 100  # samples → milliseconds

    return float(best_r), int(best_shift)

In [3]:
###############################################################################
# Main loop
###############################################################################
results = []

for sub in SUBJECTS:
    sub_tag = f"sub-{sub:03d}"
    eeg_sub = EEG_ROOT / sub_tag
    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        continue
    print(f"Processing {sub_tag}...")

    for cond_path in eeg_sub.iterdir():          # memory / control
        for load_path in cond_path.iterdir():    # 05 / 09 / 13
            eeg_epochs  = sorted(load_path.glob("trial_*.csv"))
            pupil_path  = PUPIL_ROOT / sub_tag / cond_path.name / load_path.name
            
            if not pupil_path.exists():
                print(f"Skipping {pupil_path} as it does not exist.")
                continue

            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))

            common = {e.name for e in eeg_epochs} & {p.name for p in pupil_epochs}
            if not common:
                print(f"{sub_tag} {cond_path.name} {load_path.name}: no common epochs - skipped")
                continue
            print(f"{sub_tag} {cond_path.name} {load_path.name}: {len(common)} common epochs")

            for fname in common:
                eeg_df   = pd.read_csv(load_path / fname, comment='#', skip_blank_lines=True, index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment='#', names=['time','diameter_z'], index_col=0)

                eeg = normalise_eeg(eeg_df.values)                      # (T,C)
                pupil = pupil_df['diameter_z'].values.astype(float)     # (T,)

                if len(pupil) < 40 or abs(len(pupil) - len(eeg)) > 30:
                    print(f"{sub_tag} {cond_path.name} {load_path.name} {fname}: invalid trial - skipped")
                    continue  # skip pathological trials

                r_max, lag_ms = best_cca_corr(eeg, pupil)
                results.append({
                    'subject': sub_tag,
                    'condition': cond_path.name,
                    'load': int(load_path.name),
                    'epoch': fname,
                    'r_max': r_max,
                    'lag_ms': lag_ms
                })

# -------------------------------------------------------------------------
print(f"Finished – valid trials: {len(results)}  →  saving {OUT_FILE}")
pd.DataFrame(results).to_csv(OUT_FILE, index=False)


Processing sub-033...
sub-033 control 05: 18 common epochs
(91,) (91, 1)
[[-0.85262761]
 [-0.93026962]
 [-0.50922826]
 [-0.37865261]
 [-0.63173736]
 [-1.53398933]
 [-0.61034034]
 [ 0.35179403]
 [ 1.33317013]
 [ 0.85371415]
 [-0.32388305]
 [-0.28371284]
 [-0.23634431]
 [-0.29158145]
 [-0.24431333]
 [-0.18077727]
 [ 0.09326942]
 [ 0.24655688]
 [ 0.28967794]
 [ 0.07289937]
 [ 0.12065229]
 [-0.40149254]
 [-1.95276353]
 [-2.12417013]
 [-2.29557768]
 [-2.37057482]
 [-0.58103741]
 [-0.65793644]
 [-0.52652124]
 [-0.71414414]
 [-0.45945105]
 [-0.19460018]
 [-0.12073519]
 [ 0.10633885]
 [-0.04263421]
 [ 1.48968191]
 [ 1.08523072]
 [ 0.34234958]
 [-0.38567307]
 [-0.07159768]
 [ 0.35132931]
 [ 0.88399815]
 [ 1.61856794]
 [ 0.99526418]
 [ 0.37196041]
 [-0.26406376]
 [-0.90008889]
 [-1.53611306]
 [-2.15941683]
 [-1.91606817]
 [-1.4963588 ]
 [-1.90381917]
 [-1.5443154 ]
 [-0.40444148]
 [-0.89768021]
 [-1.40098492]
 [-1.89422269]
 [-1.26347008]
 [-0.88477143]
 [-0.7466341 ]
 [-0.59331987]
 [-0.4181056

KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
"""
Per-subject Canonical Correlation Analysis (CCA) with a fixed lag
=================================================================

Goal
----
For each **subject**, find *one* temporal lag (in the range ±1 s, 100 ms steps)
that yields the highest *average* canonical correlation between multichannel
EEG (theta-band power) and pupil-diameter time series across **all valid
trials**.  Then compute the trial‑level CCA correlation **at that fixed
subject‑specific lag**, so every trial of the subject shares the same lag.

Normalisation
-------------
* **EEG** – per trial: centre each channel and scale so the *total* variance
  across time × channels equals 1 (retains inter‑channel ratios).
* **Pupil** – per trial: z‑score (mean 0, variance 1).

Outputs
-------
1. `trial_level_cca_fixedlag.csv`
   Columns: subject, condition, load, epoch, r, lag_ms
2. `subject_best_lag.csv`
   Columns: subject, lag_ms, n_trials, mean_r

Dependencies
------------
* scikit‑learn ≥ 1.0
* pandas, numpy

Run time ~10 min for the full dataset on a recent laptop.
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

OUT_TRIALS  = Path("trial_level_cca_fixedlag.csv")
OUT_SUBJECT = Path("subject_best_lag.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 98), [32, 37, 53, 61, 66, 78, 84, 94, 96])
SHIFTS = np.arange(-10, 11)      # ±1 s at 10 Hz → samples
WIN_OFFSET = 20                  # discard first/last 200 ms (~20 samples)

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000)
    cca.fit(eeg, pupil)
    # u, v = cca.transform(eeg, pupil)
    c_weights_eeg, c_weights_pupil = cca.x_weights_.flatten(), cca.y_weights_.flatten()
    combined_eeg = eeg @ c_weights_eeg
    combined_pupil = pupil * c_weights_pupil # Since pupil has 1D, just multiply

    # Calculate peraron correlation between the combinaed EEG and pupil data
    r = np.corrcoef(combined_eeg, combined_pupil)[0, 1]
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])

###############################################################################
# Main
###############################################################################

trial_rows   = []
subject_rows = []

for sub in SUBJECTS:
    sub_tag = f"sub-{sub:03d}"
    eeg_sub = EEG_ROOT / sub_tag
    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing – skipped")
        continue

    # ---------------------------------------------------------------------
    # Pass 1 – compute CCA correlations for every shift & every trial
    # ---------------------------------------------------------------------
    shift_to_corrs: dict[int, list[float]] = {s: [] for s in SHIFTS}
    trial_cache   = []  # will store (meta_dict, {shift: r})

    for cond_path in eeg_sub.iterdir():          # memory / control
        for load_path in cond_path.iterdir():    # 05 / 09 / 13
            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_path   = (PUPIL_ROOT / sub_tag / cond_path.name / load_path.name)
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))

            common = {e.name for e in eeg_epochs} & {p.name for p in pupil_epochs}
            if not common:
                print(f"{sub_tag} {cond_path.name} {load_path.name}: no common epochs - skipped")
                continue
            print(f"{sub_tag} {cond_path.name} {load_path.name}: {len(common)} common epochs")

            for fname in common:
                eeg_df   = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#", names=["time", "diameter_z"], index_col=0)

                eeg   = normalise_eeg(eeg_df.values.astype(float))
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic sanity
                if len(pupil) < 40 or abs(len(pupil) - len(eeg)) > 30:
                    print(f"{sub_tag} {cond_path.name} {load_path.name} {fname}: invalid trial - skipped")
                    continue

                # common window
                T   = min(len(eeg), len(pupil))
                win = slice(WIN_OFFSET, T - WIN_OFFSET)
                pupil_z = ((pupil - pupil.mean()) / pupil.std(ddof=0))[win].reshape(-1, 1)

                per_shift_r = {}
                for s in SHIFTS:
                    eeg_shift = np.roll(eeg, s, axis=0)[win]
                    r = cca_corr(eeg_shift, pupil_z)
                    per_shift_r[s] = r
                    shift_to_corrs[s].append(r)

                trial_cache.append((
                    {
                        "subject": sub_tag,
                        "condition": cond_path.name,
                        "load": int(load_path.name),
                        "epoch": fname
                    },
                    per_shift_r
                ))

    if not trial_cache:  # subject had no usable trials
        continue

    # ---------------------------------------------------------------------
    # Decide on the subject's best lag (max mean r over trials)
    # ---------------------------------------------------------------------
    mean_r_per_shift = {s: np.mean(vals) for s, vals in shift_to_corrs.items()}
    best_shift = max(mean_r_per_shift, key=mean_r_per_shift.get)
    best_lag_ms = best_shift * 100

    subject_rows.append({
        "subject": sub_tag,
        "lag_ms": best_lag_ms,
        "n_trials": sum(len(v) for v in shift_to_corrs.values()) // len(SHIFTS),
        "mean_r": mean_r_per_shift[best_shift]
    })

    # ---------------------------------------------------------------------
    # Pass 2 – store trial‑level r using that fixed lag
    # ---------------------------------------------------------------------
    for meta, per_shift in trial_cache:
        meta["r"] = per_shift[best_shift]
        meta["lag_ms"] = best_lag_ms
        trial_rows.append(meta)

    print(f"{sub_tag}: best lag {best_lag_ms:+d} ms (mean r={mean_r_per_shift[best_shift]:.3f})")

# -------------------------------------------------------------------------
print(f"Finished – {len(trial_rows)} trials from {len(subject_rows)} subjects")

pd.DataFrame(trial_rows).to_csv(OUT_TRIALS, index=False)
pd.DataFrame(subject_rows).to_csv(OUT_SUBJECT, index=False)
print(f"Saved {OUT_TRIALS} and {OUT_SUBJECT}")


sub-033 control 05: 18 common epochs
sub-033 control 09: 18 common epochs
sub-033 control 13: 18 common epochs
sub-033 memory 05: 35 common epochs
sub-033 memory 09: 35 common epochs
sub-033 memory 13: 36 common epochs
sub-033: best lag +0 ms (mean r=0.487)
sub-034 control 05: 15 common epochs
sub-034 control 09: 14 common epochs
sub-034 control 13: 16 common epochs
sub-034 memory 05: 35 common epochs
sub-034 memory 09: 30 common epochs
sub-034 memory 13: 32 common epochs
sub-034: best lag +0 ms (mean r=0.493)
sub-035 control 05: 5 common epochs
sub-035 control 09: 5 common epochs
sub-035 control 13: 5 common epochs
sub-035 memory 05: 11 common epochs
sub-035 memory 09: 4 common epochs
sub-035 memory 13: 1 common epochs
sub-035: best lag -100 ms (mean r=0.495)
sub-036 control 05: 10 common epochs
sub-036 control 09: 8 common epochs
sub-036 control 13: 8 common epochs
sub-036 memory 05: 14 common epochs
sub-036 memory 09: 13 common epochs
sub-036 memory 13: 16 common epochs
sub-036: bes

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
'''Are correlations larger in the memory condition than in control, ignoring which electrode they come from?”

the cleanest approach is:

Reduce each subject to one number per condition
Average the correlations across all electrodes (or across electrodes and trials) so every participant contributes exactly two scores: one for memory, one for control.

Run a paired test across subjects
A paired, one-tailed t-test (memory > control) is the simplest.
If you prefer to keep every electrode as a repeated measure you can switch to a linear-mixed model, but the conclusion is usually identical.'''
# -------- settings --------
CSV_PATH = Path("trial_level_cca.csv")   # same file as before
MIN_TRIALS = 10     # ignore subject × condition cells with fewer trials
# --------------------------

# 1) load & drop infinities / NaNs
df = pd.read_csv(CSV_PATH)
df = df[np.isfinite(df["r_max"])]

# 2) mean across *all* trials and electrodes for every subject × condition
sub_cond = (df.groupby(["subject", "condition"])["r_max"]
              .agg(["mean", "count"])
              .rename(columns={"mean":"r_mean", "count":"n_trials"}))

# 3) keep only cells with enough trials
sub_cond = sub_cond[sub_cond["n_trials"] >= MIN_TRIALS]

# 4) reshape so each row = one subject with both conditions
wide = (sub_cond.reset_index()
                   .pivot(index="subject", columns="condition", values="r_mean")
                   .dropna())      # drops subjects missing either condition

print(f"Paired subjects kept: {len(wide)}")

# 5) paired, one-tailed t-test (memory > control)
t_stat, p_two = stats.ttest_rel(wide["memory"], wide["control"])
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"\nMean r  (memory):  {wide['memory'].mean():.3f}")
print(f"Mean r  (control): {wide['control'].mean():.3f}")
print(f"Mean Δr (mem-ctl): { (wide['memory'] - wide['control']).mean():.3f}")
print(f"\nPaired t-stat   : {t_stat:.3f}")
print(f"One-tailed p     : {p_one:.4f}")

Paired subjects kept: 52

Mean r  (memory):  0.494
Mean r  (control): 0.497
Mean Δr (mem-ctl): -0.003

Paired t-stat   : -1.595
One-tailed p     : 0.9415


In [22]:
import pandas as pd
from scipy.stats import ttest_rel

df = pd.read_csv("trial_level_cca.csv")

# subject-wise means
pivot = (df.groupby(["subject", "condition"])["r_max"]
           .mean()
           .unstack())           # columns should be: control | memory
paired = pivot.dropna()

# Check the table
print(paired.round(3))           # sanity check

# Difference in the expected direction: memory − control
diff = paired["memory"] - paired["control"]

# Paired t-test (two-tailed)
t2, p2 = ttest_rel(paired["memory"], paired["control"])

# Convert to one-tailed since H1: memory > control
t_one_sided = t2
p_one_sided = p2 / 2 if diff.mean() > 0 else 1 - p2 / 2

print(f"\nMean difference (memory − control): {diff.mean():.3f}")
print(f"Two-tailed p = {p2:.4f}")
print(f"One-tailed p (memory > control) = {p_one_sided:.4f}")


condition  control  memory
subject                   
sub-033      0.507   0.492
sub-034      0.526   0.492
sub-035      0.500   0.511
sub-036      0.500   0.473
sub-038      0.486   0.503
sub-039      0.504   0.500
sub-040      0.517   0.497
sub-041      0.487   0.483
sub-042      0.491   0.489
sub-043      0.503   0.509
sub-044      0.518   0.496
sub-045      0.487   0.466
sub-046      0.428   0.553
sub-047      0.486   0.498
sub-048      0.466   0.477
sub-049      0.507   0.527
sub-050      0.514   0.515
sub-051      0.487   0.487
sub-052      0.470   0.454
sub-054      0.494   0.506
sub-055      0.490   0.475
sub-056      0.505   0.501
sub-057      0.482   0.471
sub-058      0.484   0.490
sub-059      0.488   0.493
sub-060      0.514   0.486
sub-062      0.480   0.487
sub-063      0.521   0.488
sub-064      0.489   0.473
sub-065      0.486   0.506
sub-067      0.486   0.519
sub-068      0.506   0.623
sub-069      0.543   0.550
sub-070      0.524   0.518
sub-071      0.488   0.489
s

In [27]:
'''
tests, within the memory condition only, whether the average correlation differs 
between the 5-, 9-, and 13-digit loads
'''
# ---------------------------------------------------------------
#  STEP 0 – load the per-trial correlations (if not already in `out`)
# ---------------------------------------------------------------
import pandas as pd, numpy as np
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

out = pd.read_csv("trial_level_cca_fixedlag.csv")      # skip if `out` is still in RAM

# ---------------------------------------------------------------
#  STEP 1 – keep MEMORY trials, compute a mean r per subject × load
# ---------------------------------------------------------------
mem = out[out["condition"] == "memory"]

sub_load = (mem.groupby(["subject", "load"])["r"]
              .mean()                       # mean across all trials & channels
              .unstack())                   # columns 5, 9, 13

#  keep only subjects that have *all three* loads
sub_load = sub_load.dropna()

print(f"Subjects with all loads: {len(sub_load)}")

# ---------------------------------------------------------------
#  STEP 2 – paired comparisons 5 vs 9, 5 vs 13, 9 vs 13
# ---------------------------------------------------------------
alpha = 0.05
pairs = [(5, 9), (5, 13), (9, 13)]
records = []

for a, b in pairs:
    diff = sub_load[a] - sub_load[b]        # paired difference for each subject
    t_stat, p_two = stats.ttest_rel(sub_load[a], sub_load[b])
    mean_diff = diff.mean()

    # one-tailed p: “is load a > load b ?”
    p_one = p_two / 2 if mean_diff > 0 else 1 - p_two / 2

    records.append(dict(pair=f"{a} vs {b}",
                        mean_a=sub_load[a].mean(),
                        mean_b=sub_load[b].mean(),
                        mean_diff=mean_diff,
                        t_stat=t_stat,
                        p_one=p_one))

pair_df = pd.DataFrame(records)

# ---------------------------------------------------------------
#  STEP 3 – FDR-correct across the three comparisons
# ---------------------------------------------------------------
rej, p_fdr, _, _ = multipletests(pair_df["p_one"],
                                 alpha=alpha,
                                 method="fdr_bh")
pair_df["p_FDR"]   = p_fdr
pair_df["sig_FDR"] = np.where(rej, "★", "")

# tidy formatting
pair_df = pair_df.round({"mean_a":3, "mean_b":3, "mean_diff":3,
                         "t_stat":3, "p_one":4, "p_FDR":4})

print("\nMemory-condition load comparison (one-tailed, FDR-corrected)")
print(pair_df.to_string(index=False))


Subjects with all loads: 54

Memory-condition load comparison (one-tailed, FDR-corrected)
   pair  mean_a  mean_b  mean_diff  t_stat  p_one  p_FDR sig_FDR
 5 vs 9   0.563   0.482      0.081  23.595    0.0    0.0       ★
5 vs 13   0.563   0.404      0.159  39.317    0.0    0.0       ★
9 vs 13   0.482   0.404      0.078  24.886    0.0    0.0       ★
